# Solution to p13

Assume the square patch side divides both image dimensions, so no crop or padding is needed. For side 4, the grid is `(24/4,16/4)=(6,4)` and the patch count is `N=24`. Each raw channel-first patch has `3*4*4=48` scalars, so the raw patches are `(5,24,48)`, projected patch tokens are `(5,24,32)`, the expanded class token is `(5,1,32)`, and the full token sequence is `(5,25,32)`. Per head, Q and K have shapes `(5,25,d_h)` and the matrix product `QK^T` has shape `(5,25,25)`. Therefore its scalar-entry count is exactly `B(N+1)^2=5*25^2=3125`.

For side 8, divisibility still holds: the grid is `(3,2)`, `N=6`, and the score count is `5*7^2=245`. The exact fine-to-coarse ratio is `3125/245=625/49`.

The quadratic expression describes attention-score storage (and its associated QK multiplication) as sequence length changes. It is not the projection cost: token Q/K/V projections scale linearly in token count, typically `O(B(N+1)D^2)`, while raw patch projection also depends on patch-vector size, `O(BN(Cp^2)D)`.

### Answer check

In [ ]:
from fractions import Fraction

B, C, H, W, D = 5, 3, 24, 16, 32
def token_facts(p):
    assert H % p == 0 and W % p == 0
    N = (H//p)*(W//p)
    return N, B*(N+1)**2
N4, scores4 = token_facts(4)
N8, scores8 = token_facts(8)
assert (N4, scores4) == (24, 3125)
assert (N8, scores8) == (6, 245)
assert Fraction(scores4, scores8) == Fraction(625,49)
assert (B,N4,C*4*4) == (5,24,48)
assert (B,N4+1,D) == (5,25,32)
